In [0]:
%sql
SELECT date_TRUNC('HOUR', session_start) AS session_date
, COUNT(*) AS sessions_count
, SUM(vc.session_duration)/3600.0 AS total_duration
, COUNT(DISTINCT vc.fk_commercial_id) AS commercials_count
, COUNT(DISTINCT vc.fk_tvid) AS tv_count
-- , COUNT(DISTINCT vc.prev_show_id) AS prev_show_count
-- , COUNT(DISTINCT vc.next_show_id) AS next_show_count
, COUNT(DISTINCT vc.prev_station_id) AS prev_station_count
, COUNT(DISTINCT vc.prev_vizio_epg_station) AS prev_wf_station_count
-- , COUNT(DISTINCT vc.next_station_id) AS next_station_count
-- , COUNT(DISTINCT vc.fk_location_id) AS loc_count
, COUNT(DISTINCT vc.fk_dma_id) AS dma_count
FROM prod.detection.viewing_commercials_firehose_dedup vc
JOIN prod.detection.commercial_id_external_firehose cief
  ON cief.external_id = vc.external_id
JOIN prod.detection.clients cl
  ON cl.client_id = cief.fk_client_id
WHERE vc.session_start >= CURRENT_DATE
  AND vc.fk_zoo_id = 17
  AND cl.client_name = 'kinetiq'
GROUP BY 1

In [0]:
%sql
DROP TABLE IF EXISTS dev.mohit_gangwani.ad_dma_hourly;
CREATE TABLE dev.mohit_gangwani.ad_dma_hourly AS
SELECT DATE_TRUNC('HOUR', vc.session_start) AS session_hour
, vc.external_id AS ad_id
, vc.fk_dma_id
, COUNT(DISTINCT vc.fk_tvid) AS tv_count
, COUNT(DISTINCT vc.fk_tvid||'_'||vc.session_start) AS impression_count
FROM prod.detection.viewing_commercials_firehose_dedup vc
JOIN prod.detection.commercial_id_external_firehose cief
  ON cief.external_id = vc.external_id
JOIN prod.detection.clients cl
  ON cl.client_id = cief.fk_client_id
WHERE vc.session_start >= CURRENT_DATE - 7
  AND vc.session_start < CURRENT_DATE
  AND vc.fk_zoo_id = 17
  AND cl.client_name = 'kinetiq'
  AND vc.fk_dma_id IS NOT NULL
GROUP BY 1, 2, 3;

In [0]:
%sql
SELECT COUNT(*) FROM dev.mohit_gangwani.ad_labeling_phat_10_min_binned

In [0]:
%sql
DROP TABLE IF EXISTS dev.mohit_gangwani.ad_dma_overall;
CREATE TABLE dev.mohit_gangwani.ad_dma_overall AS
SELECT vc.external_id AS ad_id
, vc.fk_dma_id
, COUNT(DISTINCT vc.fk_tvid) AS tv_count
, COUNT(DISTINCT vc.fk_tvid||'_'||vc.session_start) AS impression_count
FROM prod.detection.viewing_commercials_firehose_dedup vc
JOIN prod.detection.commercial_id_external_firehose cief
  ON cief.external_id = vc.external_id
JOIN prod.detection.clients cl
  ON cl.client_id = cief.fk_client_id
WHERE vc.session_start >= CURRENT_DATE - 7
  AND vc.session_start < CURRENT_DATE
  AND vc.fk_zoo_id = 17
  AND cl.client_name = 'kinetiq'
  AND vc.fk_dma_id IS NOT NULL
GROUP BY 1, 2
HAVING impression_count >= 20
   AND tv_count >= 10;

In [0]:
%sql
DROP TABLE IF EXISTS dev.mohit_gangwani.ad_dma_overall_for_analysis;
CREATE TABLE dev.mohit_gangwani.ad_dma_overall_for_analysis AS
SELECT vc.external_id AS ad_id
, vc.fk_dma_id
, COUNT(DISTINCT vc.fk_tvid) AS tv_count
, COUNT(DISTINCT vc.fk_tvid||'_'||vc.session_start) AS impression_count
FROM prod.detection.viewing_commercials_firehose_dedup vc
JOIN prod.detection.commercial_id_external_firehose cief
  ON cief.external_id = vc.external_id
JOIN prod.detection.clients cl
  ON cl.client_id = cief.fk_client_id
WHERE vc.session_start >= CURRENT_DATE - 7
  AND vc.session_start < CURRENT_DATE
  AND vc.fk_zoo_id = 17
  AND cl.client_name = 'kinetiq'
  AND vc.fk_dma_id IS NOT NULL
GROUP BY 1, 2

In [0]:
%sql
DROP TABLE IF EXISTS dev.mohit_gangwani.ad_dma_overall_for_analysis_part_two;
CREATE TABLE dev.mohit_gangwani.ad_dma_overall_for_analysis_part_two AS
SELECT ad_id
, COUNT(*) AS dmas_ge_1
, COUNT_IF(impression_count >= 5) AS dmas_ge_5
, COUNT_IF(impression_count >= 10) AS dmas_ge_10
, COUNT_IF(impression_count >= 20) AS dmas_ge_20
, COUNT_IF(impression_count >= 30) AS dmas_ge_30
, COUNT_IF(impression_count >= 40) AS dmas_ge_40
, COUNT_IF(impression_count >= 50) AS dmas_ge_50
FROM dev.mohit_gangwani.ad_dma_overall_for_analysis
GROUP BY 1

In [0]:
%sql
SELECT *
FROM dev.mohit_gangwani.ad_dma_overall_for_analysis_part_two
ORDER BY 2 DESC
LIMIT 1000

In [0]:
%sql
SELECT 'ge_1' AS threshold_label
, 1 AS threshhold_val
, PERCENTILE(dmas_ge_1, 0.50) AS p50_dmas
, PERCENTILE(dmas_ge_1, 0.75) AS p75_dmas
FROM dev.mohit_gangwani.ad_dma_overall_for_analysis_part_two
UNION ALL
SELECT 'ge_5' AS threshold_label
, 5 AS threshhold_val
, PERCENTILE(dmas_ge_5, 0.50) AS p50_dmas
, PERCENTILE(dmas_ge_5, 0.75) AS p75_dmas
FROM dev.mohit_gangwani.ad_dma_overall_for_analysis_part_two
UNION ALL
SELECT 'ge_10' AS threshold_label
, 10 AS threshhold_val
, PERCENTILE(dmas_ge_10, 0.50) AS p50_dmas
, PERCENTILE(dmas_ge_10, 0.75) AS p75_dmas
FROM dev.mohit_gangwani.ad_dma_overall_for_analysis_part_two
UNION ALL
SELECT 'ge_20' AS threshold_label
, 20 AS threshhold_val
, PERCENTILE(dmas_ge_20, 0.50) AS p50_dmas
, PERCENTILE(dmas_ge_20, 0.75) AS p75_dmas
FROM dev.mohit_gangwani.ad_dma_overall_for_analysis_part_two
UNION ALL
SELECT 'ge_30' AS threshold_label
, 30 AS threshhold_val
, PERCENTILE(dmas_ge_30, 0.50) AS p50_dmas
, PERCENTILE(dmas_ge_30, 0.75) AS p75_dmas
FROM dev.mohit_gangwani.ad_dma_overall_for_analysis_part_two
UNION ALL
SELECT 'ge_40' AS threshold_label
, 40 AS threshhold_val
, PERCENTILE(dmas_ge_40, 0.50) AS p50_dmas
, PERCENTILE(dmas_ge_40, 0.75) AS p75_dmas
FROM dev.mohit_gangwani.ad_dma_overall_for_analysis_part_two
UNION ALL
SELECT 'ge_50' AS threshold_label
, 50 AS threshhold_val
, PERCENTILE(dmas_ge_50, 0.50) AS p50_dmas
, PERCENTILE(dmas_ge_50, 0.75) AS p75_dmas
FROM dev.mohit_gangwani.ad_dma_overall_for_analysis_part_two

In [0]:
import matplotlib.pyplot as plt

thresholds = [1, 5, 10, 20, 30, 40, 50]
p50_dmas = [63, 6, 2, 1, 1, 0, 0]
p75_dmas = [185, 100, 54, 22, 11, 7, 5]

plt.figure(figsize=(7,4))
plt.plot(thresholds, p50_dmas, marker='o', label='Median DMAs per ad (p50)')
plt.plot(thresholds, p75_dmas, marker='s', label='75th percentile DMAs per ad (p75)')
plt.xlabel("Impression threshold per AdxDMA")
plt.ylabel("DMAs per ad")
plt.title("Elbow Curve: DMAs per Ad vs Impression Threshold")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

In [0]:
%sql
DROP TABLE IF EXISTS dev.mohit_gangwani.ad_dma_overall_for_analysis_part_three;
CREATE TABLE dev.mohit_gangwani.ad_dma_overall_for_analysis_part_three AS
SELECT ad_id
, COUNT(*) AS dmas_ge_1
, COUNT_IF(tv_count >= 5) AS dmas_ge_5
, COUNT_IF(tv_count >= 10) AS dmas_ge_10
, COUNT_IF(tv_count >= 20) AS dmas_ge_20
, COUNT_IF(tv_count >= 30) AS dmas_ge_30
, COUNT_IF(tv_count >= 40) AS dmas_ge_40
, COUNT_IF(tv_count >= 50) AS dmas_ge_50
FROM dev.mohit_gangwani.ad_dma_overall_for_analysis
GROUP BY 1

In [0]:
%sql
SELECT 'ge_1' AS threshold_label
, 1 AS threshhold_val
, PERCENTILE(dmas_ge_1, 0.50) AS p50_dmas
, PERCENTILE(dmas_ge_1, 0.75) AS p75_dmas
FROM dev.mohit_gangwani.ad_dma_overall_for_analysis_part_three
UNION ALL
SELECT 'ge_5' AS threshold_label
, 5 AS threshhold_val
, PERCENTILE(dmas_ge_5, 0.50) AS p50_dmas
, PERCENTILE(dmas_ge_5, 0.75) AS p75_dmas
FROM dev.mohit_gangwani.ad_dma_overall_for_analysis_part_three
UNION ALL
SELECT 'ge_10' AS threshold_label
, 10 AS threshhold_val
, PERCENTILE(dmas_ge_10, 0.50) AS p50_dmas
, PERCENTILE(dmas_ge_10, 0.75) AS p75_dmas
FROM dev.mohit_gangwani.ad_dma_overall_for_analysis_part_three
UNION ALL
SELECT 'ge_20' AS threshold_label
, 20 AS threshhold_val
, PERCENTILE(dmas_ge_20, 0.50) AS p50_dmas
, PERCENTILE(dmas_ge_20, 0.75) AS p75_dmas
FROM dev.mohit_gangwani.ad_dma_overall_for_analysis_part_three
UNION ALL
SELECT 'ge_30' AS threshold_label
, 30 AS threshhold_val
, PERCENTILE(dmas_ge_30, 0.50) AS p50_dmas
, PERCENTILE(dmas_ge_30, 0.75) AS p75_dmas
FROM dev.mohit_gangwani.ad_dma_overall_for_analysis_part_three
UNION ALL
SELECT 'ge_40' AS threshold_label
, 40 AS threshhold_val
, PERCENTILE(dmas_ge_40, 0.50) AS p50_dmas
, PERCENTILE(dmas_ge_40, 0.75) AS p75_dmas
FROM dev.mohit_gangwani.ad_dma_overall_for_analysis_part_three
UNION ALL
SELECT 'ge_50' AS threshold_label
, 50 AS threshhold_val
, PERCENTILE(dmas_ge_50, 0.50) AS p50_dmas
, PERCENTILE(dmas_ge_50, 0.75) AS p75_dmas
FROM dev.mohit_gangwani.ad_dma_overall_for_analysis_part_three

In [0]:
thresholds = [1, 5, 10, 20, 30, 40, 50]
p50_dmas = [63, 5, 2, 1, 1, 0, 0]
p75_dmas = [185, 98, 52, 20, 10, 7, 5]

plt.figure(figsize=(7,4))
plt.plot(thresholds, p50_dmas, marker='o', label='Median DMAs per ad (p50)')
plt.plot(thresholds, p75_dmas, marker='s', label='75th percentile DMAs per ad (p75)')
plt.xlabel("TV Count threshold per AdxDMA")
plt.ylabel("DMAs per ad")
plt.title("Elbow Curve: DMAs per Ad vs TV Count Threshold")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

In [0]:
%sql
SELECT dma_count, impressions_per_dma, COUNT(DISTINCT ad_id) AS ad_count
FROM (
  SELECT ad_id, COUNT(DISTINCT fk_dma_id) AS dma_count, SUM(impression_count) AS ttl_impressions, ROUND(ttl_impressions/dma_count, 0) AS impressions_per_dma
  FROM dev.mohit_gangwani.ad_dma_overall_for_analysis
  GROUP BY 1
)
GROUP BY 1, 2
HAVING ad_count > 2
ORDER BY 1, 2

In [0]:
%sql
SELECT CASE WHEN impression_count <= 100 THEN impression_count
            ELSE 100
            END AS impression_count_grp
, COUNT(DISTINCT ad_id) AS ad_count
FROM dev.mohit_gangwani.ad_dma_overall_for_analysis
GROUP BY 1
ORDER BY 1

In [0]:
%sql
SELECT CASE WHEN ttl_impressions < 5 THEN '<5'
            WHEN ttl_impressions < 10 THEN '5-10'
            WHEN ttl_impressions < 20 THEN '10-20'
            WHEN ttl_impressions < 50 THEN '20-50'
            ELSE '>50'
            END AS impression_count_grp
, CASE WHEN ttl_impressions < 5 THEN 1
       WHEN ttl_impressions < 10 THEN 2
       WHEN ttl_impressions < 20 THEN 3
       WHEN ttl_impressions < 50 THEN 4
       ELSE 5
       END AS impression_count_so
, COUNT(*) AS ad_count
FROM (
  SELECT ad_id, fk_dma_id, SUM(impression_count) AS ttl_impressions
  FROM dev.mohit_gangwani.ad_dma_overall_for_analysis
  GROUP BY 1, 2
  )
GROUP BY 1, 2
ORDER BY 2

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
%sql
SELECT CASE WHEN impressions_per_dma = 1 THEN '1'
            WHEN impressions_per_dma < 5 THEN '1-4'
            WHEN impressions_per_dma < 10 THEN '5-9'
            WHEN impressions_per_dma < 20 THEN '10-19'
            WHEN impressions_per_dma < 50 THEN '20-49'
            WHEN impressions_per_dma < 100 THEN '50-100'
            ELSE '>100'
            END AS impressions_per_dma_grp
, CASE WHEN total_dmas = 1 THEN '1'
       WHEN total_dmas < 5 THEN '1-4'
       WHEN total_dmas < 10 THEN '5-9'
       WHEN total_dmas < 20 THEN '10-19'
       WHEN total_dmas < 50 THEN '20-49'
       WHEN total_dmas < 100 THEN '50-100'
       ELSE '>100'
       END AS dma_count_grp
, COUNT(*)
FROM (
  SELECT ad_id, COUNT(DISTINCT fk_dma_id) AS total_dmas, ROUND(SUM(impression_count)/COUNT(DISTINCT fk_dma_id), 0) AS impressions_per_dma
  FROM dev.mohit_gangwani.ad_dma_overall_for_analysis
  GROUP BY 1
  )
GROUP BY 1, 2
ORDER BY 1, 2

In [0]:
%sql
SELECT percentile(impressions_per_dma, 0.50) AS p50
, percentile(impressions_per_dma, 0.75) AS p75
, percentile(impressions_per_dma, 0.90) AS p90
, percentile(impressions_per_dma, 0.95) AS p95
FROM (
  SELECT ad_id
  , fk_dma_id
  , SUM(impression_count) AS impressions_per_dma
  -- , ROUND(SUM(impression_count)/COUNT(DISTINCT fk_dma_id), 0) AS impressions_per_dma
  FROM dev.mohit_gangwani.ad_dma_overall_for_analysis
  GROUP BY 1, 2
)

In [0]:
%sql
SELECT impressions_per_dma
, COUNT(*) AS dma_count
FROM (
  SELECT fk_dma_id, SUM(impression_count) AS impressions_per_dma
  FROM dev.mohit_gangwani.ad_dma_overall_for_analysis
  GROUP BY 1
  )
GROUP BY 1
ORDER BY 1

In [0]:
%sql
SELECT CASE WHEN tv_count <= 10 THEN '<=10'
            WHEN tv_count <= 50 THEN '11-50'
            WHEN tv_count <= 100 THEN '51-100'
            WHEN tv_count <= 200 THEN '101-200'
            WHEN tv_count <= 500 THEN '201-500'
            WHEN tv_count <= 1000 THEN '5001-1000'
            WHEN tv_count <= 2000 THEN '1001-2000'
            WHEN tv_count <= 5000 THEN '2001-5000'
            ELSE '>5000'
  END AS tv_count_grp
, CASE WHEN tv_count <= 10 THEN 1
       WHEN tv_count <= 50 THEN 2
       WHEN tv_count <= 100 THEN 3
       WHEN tv_count <= 200 THEN 4
       WHEN tv_count <= 500 THEN 5
       WHEN tv_count <= 1000 THEN 6
       WHEN tv_count <= 2000 THEN 7
       WHEN tv_count <= 5000 THEN 8
       ELSE 9
  END AS tv_count_so
, COUNT(DISTINCT ad_id) AS ad_count
FROM dev.mohit_gangwani.ad_dma_overall_for_analysis
GROUP BY 1, 2
ORDER BY 2

In [0]:
%sql
DROP TABLE IF EXISTS dev.mohit_gangwani.dma_opportunities_hourly;
CREATE TABLE dev.mohit_gangwani.dma_opportunities_hourly AS
SELECT DATE_TRUNC('HOUR', vc.session_start) AS session_hour
, vc.fk_dma_id
, COUNT(DISTINCT vc.fk_tvid) AS active_tvs
FROM prod.detection.viewing_commercials_firehose_dedup vc
JOIN prod.detection.commercial_id_external_firehose cief
  ON cief.external_id = vc.external_id
JOIN prod.detection.clients cl
  ON cl.client_id = cief.fk_client_id
WHERE vc.session_start >= CURRENT_DATE - 7
  AND vc.session_start < CURRENT_DATE
  AND vc.fk_zoo_id = 17
  AND cl.client_name = 'kinetiq'
GROUP BY 1, 2;

In [0]:
%sql
DROP TABLE IF EXISTS dev.mohit_gangwani.ad_dmas_count;
CREATE TABLE dev.mohit_gangwani.ad_dmas_count AS
WITH
-- ad_filter AS (
--   SELECT ad_id, impression_count, tv_count
--   FROM dev.mohit_gangwani.ad_dma_overall
--   WHERE impression_count >= 20
--     AND tv_count >= 10
--   GROUP BY ALL
-- )
-- , 
ad_dma AS (
  SELECT a.ad_id, a.fk_dma_id, a.impression_count, a.tv_count
  FROM dev.mohit_gangwani.ad_dma_overall a
  -- JOIN ad_filter f
  --   ON a.ad_id = f.ad_id
  GROUP BY ALL
)
SELECT a.ad_id
, COUNT(DISTINCT a.fk_dma_id) AS dma_count
, SUM(a.impression_count) AS ttl_impressions
, SUM(a.tv_count) AS tv_count
, dma_count*1.0/ 210 AS dma_coverage
FROM ad_dma a
GROUP BY 1;

In [0]:
%sql
SELECT * FROM dev.mohit_gangwani.ad_dmas_count
WHERE dma_count <= 2
ORDER BY ttl_impressions DESC
LIMIT 10

In [0]:
%sql
SELECT * FROM dev.mohit_gangwani.ad_dma_overall
WHERE ad_id = 'AE16546-2025-30-04769'

In [0]:
%sql
SELECT * FROM dev.mohit_gangwani.ad_dmas_count
WHERE dma_count >= 100
ORDER BY ttl_impressions DESC
LIMIT 10

In [0]:
%sql
SELECT * FROM dev.mohit_gangwani.ad_dmas_count
WHERE dma_count >= 3
  AND dma_count < 100
ORDER BY ttl_impressions DESC
LIMIT 10

In [0]:
%sql
SELECT * FROM dev.mohit_gangwani.ad_dma_overall
WHERE ad_id IN ('AE16546-2025-43-05892', 'AE16546-2024-45-01430', 'AE16546-2025-43-07032', 'AE16546-2025-43-05083', 'AE16546-2025-43-05730', 'AE16546-2025-43-04818', 'AE16546-2025-41-03971', 'AE16546-2023-42-05587', 'AE16546-2025-09-01406', 'AE16546-2024-42-01939')

In [0]:
%sql
SELECT APPROX_PERCENTILE(dma_count, 0.25) AS perc_25
, APPROX_PERCENTILE(dma_count, 0.50) AS perc_50
, APPROX_PERCENTILE(dma_count, 0.75) AS perc_75
, APPROX_PERCENTILE(dma_count, 0.90) AS perc_90
, AVG(dma_count) AS avg_dma_count
FROM dev.mohit_gangwani.ad_dmas_count

In [0]:
%sql
SELECT dma_count
, COUNT(DISTINCT ad_id)
FROM dev.mohit_gangwani.ad_dmas_count
GROUP BY 1

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
SELECT COUNT(DISTINCT ad_id)
FROM dev.mohit_gangwani.ad_dmas_count

In [0]:
import pandas as pd
import numpy as np

In [0]:
def gini(ad):
    x = np.sort(np.array(ad))
    n = x.size
    if n <= 1 or np.sum(x) == 0:
        return 1.0

    index = np.arange(1, n + 1)
    a = (2.0 * np.sum(index * x))/(n * np.sum(x))
    c = (n + 1) / n
    g = a - c
    norm = n / (n - 1)
    return norm * g

In [0]:
def locality_index(values):
    if len(values) <= 1 or np.sum(values) == 0:
        return 1.0
    p = np.array(values) / np.sum(values)
    p = p[p > 0]
    H = -np.sum(p * np.log(p))
    return 1 - (H / np.log(len(p)))

In [0]:
def get_gini_entropy_df(df, ad_ids):
    x_df = df[df.ad_id.isin(ad_ids)].copy()
    x_df.sort_values(by=['ad_id', 'impression_count'], inplace=True)
    x_df.reset_index(inplace=True, drop=True)
    ndf = pd.DataFrame()
    for i, ad_id in enumerate(ad_ids):
        a = x_df[x_df['ad_id'] == ad_id]
        ndf.loc[i, 'ad_id'] = ad_id
        ndf.loc[i, 'impression_count'] = a.impression_count.sum()
        ndf.loc[i, 'dma_count'] = a.fk_dma_id.nunique()
        ndf.loc[i, 'gini'] = gini(a['impression_count'])
        ndf.loc[i, 'entropy'] = locality_index(a['impression_count'])
    return ndf

In [0]:
spark.sql(f"""
          SELECT inscape_station_id, mapped_vendor, COUNT(*)
          FROM (
      SELECT 
        inscape_station_id, 
        inscape_call_sign, 
        mapped_vendor,
        mapped_vendor_station_id
    FROM (
        SELECT 
          inscape_station_id, 
          inscape_call_sign, 
          mapped_vendor, 
          mapped_vendor_station_id,
          ROW_NUMBER() OVER (PARTITION BY mapped_vendor, mapped_vendor_station_id ORDER BY created_at DESC) AS rn
        --   _one,
        --   ROW_NUMBER() OVER (PARTITION BY mapped_vendor, inscape_station_id ORDER BY created_at DESC) AS rn_two
        FROM prod.detection.inscape_station_map
      ) ism
    WHERE ism.rn = 1)
    GROUP BY 1, 2
    HAVING COUNT(*) > 1
  """).display()